# MRI Preprocessing Pipeline: 3T to 7T Super-Resolution

This notebook implements a comprehensive preprocessing pipeline for paired 3T and 7T brain MRI data.

## Pipeline Overview
1. **BIDS Validation** - Validate dataset structure
2. **Session Separation** - Separate 3T and 7T data by field strength
3. **Skull Stripping** - Remove non-brain tissue using SynthStrip/HD-BET
4. **Bias Field Correction** - N4 bias correction
5. **Registration** - Align 3T to 7T space
6. **Resolution Harmonization** - Resample to common grid
7. **Intensity Normalization** - Z-score normalization
8. **Tissue Segmentation** - GM/WM/CSF segmentation (optional)

---

## Setup: Import Libraries and Configure Paths

## Configuration (Optional)

Customize paths and parameters here. Leave as `None` to use auto-detected defaults.

In [ ]:
# ============================================================================
# USER CONFIGURATION (Optional)
# ============================================================================
# Set these to override auto-detection. Leave as None to use defaults.

# Custom project root directory (will auto-detect if None)
CUSTOM_BASE_DIR = None  # e.g., Path("/path/to/Topo-Brain")

# Custom raw data directory name (default: "UNC")
RAW_DATA_DIRNAME = "UNC"

# Custom output directory name (default: "processed")
OUTPUT_DIRNAME = "processed"

# Processing parameters
N4_ITERATIONS = 50  # Number of N4 bias correction iterations
N4_CONVERGENCE = 0.001  # Convergence threshold for N4

# Target spacing for resolution harmonization (None = use 7T native)
TARGET_SPACING = None  # e.g., (0.7, 0.7, 0.7) to force specific spacing

# Normalization method: "zscore" or "percentile"
NORMALIZATION_METHOD = "zscore"

print("✓ Configuration loaded")

In [ ]:
import os
import json
import shutil
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import nibabel as nib
import SimpleITK as sitk
from tqdm.auto import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage

# ============================================================================
# AUTO-DETECT PROJECT PATHS
# ============================================================================

def detect_project_root() -> Path:
    """
    Intelligently detect the project root directory.
    Tries multiple methods:
    1. Check if TOPOBRAIN_ROOT environment variable is set
    2. Use custom base dir if provided
    3. Walk up from current directory to find project root markers
    4. Fall back to current directory
    """
    # Method 1: Environment variable
    if "TOPOBRAIN_ROOT" in os.environ:
        return Path(os.environ["TOPOBRAIN_ROOT"]).resolve()
    
    # Method 2: Custom configuration
    if 'CUSTOM_BASE_DIR' in globals() and CUSTOM_BASE_DIR is not None:
        return Path(CUSTOM_BASE_DIR).resolve()
    
    # Method 3: Look for project root markers
    current = Path.cwd().resolve()
    project_markers = ['requirements.txt', 'README.md', '.git', 'setup.py', 'pyproject.toml']
    
    # Walk up the directory tree
    for parent in [current] + list(current.parents):
        # Check if any marker exists
        if any((parent / marker).exists() for marker in project_markers):
            # Extra validation: check if it has expected structure
            if (parent / 'notebooks').exists() or (parent / 'src').exists():
                return parent
    
    # Method 4: Fall back to parent of current directory if in notebooks/
    if current.name == "notebooks" or (current.parent / "notebooks").exists():
        return current.parent if current.name == "notebooks" else current
    
    # Last resort: current directory
    return current

# Detect base directory
BASE_DIR = detect_project_root()

# Set up data paths relative to base directory
RAW_DATA_DIRNAME = RAW_DATA_DIRNAME if 'RAW_DATA_DIRNAME' in globals() else "UNC"
OUTPUT_DIRNAME = OUTPUT_DIRNAME if 'OUTPUT_DIRNAME' in globals() else "processed"

DATA_DIR = BASE_DIR / "data"
RAW_DATA_DIR = DATA_DIR / RAW_DATA_DIRNAME
PROCESSED_DIR = DATA_DIR / OUTPUT_DIRNAME
TEMP_DIR = DATA_DIR / "temp"

# Create directories
for dir_path in [PROCESSED_DIR, TEMP_DIR, PROCESSED_DIR / "3T", PROCESSED_DIR / "7T"]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Display configuration
print("="*70)
print("PREPROCESSING PIPELINE - PATH CONFIGURATION")
print("="*70)
print(f"✓ Project root:     {BASE_DIR}")
print(f"✓ Data directory:   {DATA_DIR}")
print(f"✓ Raw data (input): {RAW_DATA_DIR}")
print(f"✓ Output directory: {PROCESSED_DIR}")
print(f"✓ Temp directory:   {TEMP_DIR}")
print("="*70)

# Validation
if not RAW_DATA_DIR.exists():
    print(f"\n⚠ WARNING: Raw data directory not found: {RAW_DATA_DIR}")
    print(f"   Please ensure your BIDS data is located at this path.")
    print(f"   Or set CUSTOM_BASE_DIR in the configuration cell.")
else:
    print(f"\n✓ Raw data directory exists")

print(f"\n💡 TIP: Set TOPOBRAIN_ROOT environment variable to override auto-detection")
print(f"   Example: os.environ['TOPOBRAIN_ROOT'] = '/custom/path'")

## Step 5.1: BIDS Validation

Validate the BIDS structure and check for naming inconsistencies.

In [ ]:
def validate_bids_structure(bids_dir: Path) -> Dict[str, any]:
    """
    Validate BIDS structure and collect dataset information.
    """
    validation_results = {
        "subjects": [],
        "sessions": [],
        "issues": [],
        "statistics": {}
    }
    
    if not bids_dir.exists():
        validation_results["issues"].append(f"BIDS directory not found: {bids_dir}")
        return validation_results
    
    # Find all subjects
    subject_dirs = sorted([d for d in bids_dir.iterdir() if d.is_dir() and d.name.startswith("sub-")])
    
    print(f"Found {len(subject_dirs)} subjects")
    
    for sub_dir in subject_dirs:
        subject_id = sub_dir.name
        validation_results["subjects"].append(subject_id)
        
        # Find sessions
        session_dirs = sorted([d for d in sub_dir.iterdir() if d.is_dir() and d.name.startswith("ses-")])
        
        for ses_dir in session_dirs:
            session_id = ses_dir.name
            validation_results["sessions"].append(f"{subject_id}/{session_id}")
            
            # Check for anatomical images
            anat_dir = ses_dir / "anat"
            if not anat_dir.exists():
                validation_results["issues"].append(f"Missing anat directory: {subject_id}/{session_id}")
                continue
            
            # Find NIfTI files
            nii_files = list(anat_dir.glob("*.nii*"))
            json_files = list(anat_dir.glob("*.json"))
            
            if not nii_files:
                validation_results["issues"].append(f"No NIfTI files in: {subject_id}/{session_id}/anat")
            
            # Check for corresponding JSON files
            for nii_file in nii_files:
                json_file = nii_file.parent / f"{nii_file.name.split('.nii')[0]}.json"
                if not json_file.exists():
                    validation_results["issues"].append(f"Missing JSON for: {nii_file.relative_to(bids_dir)}")
    
    validation_results["statistics"] = {
        "total_subjects": len(validation_results["subjects"]),
        "total_sessions": len(validation_results["sessions"]),
        "total_issues": len(validation_results["issues"])
    }
    
    return validation_results

# Run validation
print("Running BIDS validation...")
validation_results = validate_bids_structure(RAW_DATA_DIR)

print(f"\n{'='*60}")
print(f"BIDS VALIDATION RESULTS")
print(f"{'='*60}")
print(f"✓ Subjects found: {validation_results['statistics']['total_subjects']}")
print(f"✓ Sessions found: {validation_results['statistics']['total_sessions']}")
print(f"⚠ Issues found: {validation_results['statistics']['total_issues']}")

if validation_results["issues"]:
    print(f"\n{'='*60}")
    print("ISSUES DETECTED:")
    print(f"{'='*60}")
    for issue in validation_results["issues"][:10]:  # Show first 10
        print(f"  • {issue}")
    if len(validation_results["issues"]) > 10:
        print(f"  ... and {len(validation_results['issues']) - 10} more issues")
else:
    print("\n✓ No issues detected - BIDS structure is valid!")

## Step 5.2: Separate 3T and 7T Sessions

Parse JSON metadata to extract field strength and organize data by scanner type.

In [ ]:
def get_field_strength(json_path: Path) -> Optional[float]:
    """
    Extract magnetic field strength from JSON sidecar.
    """
    try:
        with open(json_path, 'r') as f:
            metadata = json.load(f)
            return metadata.get("MagneticFieldStrength", None)
    except Exception as e:
        print(f"Error reading {json_path}: {e}")
        return None

def separate_by_field_strength(bids_dir: Path, output_dir: Path) -> Dict[str, List]:
    """
    Separate data by field strength (3T vs 7T).
    Creates organized directory structure and copies/links files.
    """
    field_strength_data = {
        "3T": [],
        "7T": [],
        "unknown": []
    }
    
    # Find all subjects
    subject_dirs = sorted([d for d in bids_dir.iterdir() if d.is_dir() and d.name.startswith("sub-")])
    
    print(f"Processing {len(subject_dirs)} subjects...")
    
    for sub_dir in tqdm(subject_dirs, desc="Separating data by field strength"):
        subject_id = sub_dir.name
        
        # Find sessions
        session_dirs = sorted([d for d in sub_dir.iterdir() if d.is_dir() and d.name.startswith("ses-")])
        
        for ses_dir in session_dirs:
            session_id = ses_dir.name
            anat_dir = ses_dir / "anat"
            
            if not anat_dir.exists():
                continue
            
            # Find all JSON files
            json_files = list(anat_dir.glob("*.json"))
            
            for json_file in json_files:
                field_strength = get_field_strength(json_file)
                
                # Classify by field strength
                if field_strength is not None:
                    if 2.5 <= field_strength <= 3.5:
                        category = "3T"
                    elif 6.5 <= field_strength <= 7.5:
                        category = "7T"
                    else:
                        category = "unknown"
                else:
                    category = "unknown"
                
                # Get corresponding NIfTI file
                nii_base = json_file.name.replace(".json", "")
                nii_file = None
                for ext in [".nii.gz", ".nii"]:
                    candidate = anat_dir / f"{nii_base}{ext}"
                    if candidate.exists():
                        nii_file = candidate
                        break
                
                if nii_file:
                    field_strength_data[category].append({
                        "subject": subject_id,
                        "session": session_id,
                        "nii_file": nii_file,
                        "json_file": json_file,
                        "field_strength": field_strength
                    })
    
    # Create organized directory structure
    for category in ["3T", "7T"]:
        category_dir = output_dir / category
        category_dir.mkdir(parents=True, exist_ok=True)
        
        # Group by subject
        subjects = {}
        for item in field_strength_data[category]:
            subj = item["subject"]
            if subj not in subjects:
                subjects[subj] = []
            subjects[subj].append(item)
        
        # Create subject directories and copy files
        for subject_id, items in subjects.items():
            subj_dir = category_dir / subject_id / "anat"
            subj_dir.mkdir(parents=True, exist_ok=True)
            
            for item in items:
                # Copy NIfTI file
                dest_nii = subj_dir / item["nii_file"].name
                if not dest_nii.exists():
                    shutil.copy2(item["nii_file"], dest_nii)
                
                # Copy JSON file
                dest_json = subj_dir / item["json_file"].name
                if not dest_json.exists():
                    shutil.copy2(item["json_file"], dest_json)
    
    return field_strength_data

# Execute separation
print("Separating data by field strength...")
field_strength_data = separate_by_field_strength(RAW_DATA_DIR, PROCESSED_DIR)

# Print summary
print(f"\n{'='*60}")
print(f"FIELD STRENGTH SEPARATION RESULTS")
print(f"{'='*60}")
print(f"✓ 3T scans: {len(field_strength_data['3T'])}")
print(f"✓ 7T scans: {len(field_strength_data['7T'])}")
print(f"⚠ Unknown: {len(field_strength_data['unknown'])}")

# Create a summary dataframe
summary_data = []
for category in ["3T", "7T", "unknown"]:
    for item in field_strength_data[category]:
        summary_data.append({
            "Category": category,
            "Subject": item["subject"],
            "Session": item["session"],
            "Field Strength": item.get("field_strength", "N/A"),
            "File": item["nii_file"].name
        })

summary_df = pd.DataFrame(summary_data)
print(f"\nFirst 10 entries:")
print(summary_df.head(10).to_string(index=False))

## Step 5.3: Skull Stripping with SynthStrip

Use modern deep learning-based skull stripping (SynthStrip from FreeSurfer or HD-BET).

**Note:** This implementation provides both a pure Python fallback and wrapper for SynthStrip command-line tool.

In [ ]:
def skull_strip_synthstrip(input_path: Path, output_path: Path, mask_path: Path = None) -> bool:
    """
    Perform skull stripping using SynthStrip (if available).
    Falls back to robust threshold-based method if SynthStrip is not installed.
    """
    # Check if SynthStrip is available
    try:
        result = subprocess.run(["mri_synthstrip", "--help"], 
                              capture_output=True, timeout=5)
        synthstrip_available = result.returncode == 0
    except:
        synthstrip_available = False
    
    if synthstrip_available:
        # Use SynthStrip
        cmd = ["mri_synthstrip", "-i", str(input_path), "-o", str(output_path)]
        if mask_path:
            cmd.extend(["-m", str(mask_path)])
        
        try:
            subprocess.run(cmd, check=True, capture_output=True)
            return True
        except subprocess.CalledProcessError as e:
            print(f"SynthStrip failed: {e}")
            return False
    else:
        # Fallback: Use robust threshold-based skull stripping
        print("SynthStrip not available, using fallback method...")
        return skull_strip_fallback(input_path, output_path, mask_path)

def skull_strip_fallback(input_path: Path, output_path: Path, mask_path: Path = None) -> bool:
    """
    Robust fallback skull stripping using morphological operations and thresholding.
    """
    try:
        # Load image
        img = nib.load(input_path)
        data = img.get_fdata()
        
        # Normalize intensities
        data_norm = (data - np.min(data)) / (np.max(data) - np.min(data) + 1e-8)
        
        # Otsu-like thresholding
        threshold = np.percentile(data_norm[data_norm > 0], 20)
        brain_mask = data_norm > threshold
        
        # Morphological operations to clean up mask
        from scipy.ndimage import binary_erosion, binary_dilation, binary_fill_holes
        
        # Fill holes
        brain_mask = binary_fill_holes(brain_mask)
        
        # Erosion to remove thin connections
        brain_mask = binary_erosion(brain_mask, iterations=2)
        
        # Keep only largest connected component
        from scipy.ndimage import label
        labeled, num_features = label(brain_mask)
        if num_features > 0:
            sizes = np.bincount(labeled.ravel())[1:]  # Skip background
            largest_label = np.argmax(sizes) + 1
            brain_mask = labeled == largest_label
        
        # Dilate back to recover brain boundary
        brain_mask = binary_dilation(brain_mask, iterations=3)
        
        # Apply mask
        brain_data = data * brain_mask
        
        # Save brain image
        brain_img = nib.Nifti1Image(brain_data, img.affine, img.header)
        nib.save(brain_img, output_path)
        
        # Save mask if requested
        if mask_path:
            mask_img = nib.Nifti1Image(brain_mask.astype(np.uint8), img.affine, img.header)
            nib.save(mask_img, mask_path)
        
        return True
    except Exception as e:
        print(f"Fallback skull stripping failed: {e}")
        return False

def process_skull_stripping(data_dict: Dict, category: str) -> List[Dict]:
    """
    Process skull stripping for all scans in a category.
    """
    processed_items = []
    items = data_dict[category]
    
    output_dir = PROCESSED_DIR / category / "skull_stripped"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\nProcessing {len(items)} {category} scans...")
    
    for item in tqdm(items, desc=f"Skull stripping {category}"):
        subject = item["subject"]
        input_file = item["nii_file"]
        
        # Create output paths
        subj_dir = output_dir / subject
        subj_dir.mkdir(parents=True, exist_ok=True)
        
        output_file = subj_dir / f"{input_file.stem}_brain.nii.gz"
        mask_file = subj_dir / f"{input_file.stem}_brain_mask.nii.gz"
        
        # Skip if already processed
        if output_file.exists():
            processed_items.append({
                **item,
                "brain_file": output_file,
                "mask_file": mask_file if mask_file.exists() else None
            })
            continue
        
        # Perform skull stripping
        success = skull_strip_synthstrip(input_file, output_file, mask_file)
        
        if success:
            processed_items.append({
                **item,
                "brain_file": output_file,
                "mask_file": mask_file if mask_file.exists() else None
            })
        else:
            print(f"Failed to process: {input_file}")
    
    return processed_items

# Process both 3T and 7T data
print("="*60)
print("SKULL STRIPPING")
print("="*60)

processed_3T = process_skull_stripping(field_strength_data, "3T")
processed_7T = process_skull_stripping(field_strength_data, "7T")

print(f"\n✓ Successfully processed {len(processed_3T)} 3T scans")
print(f"✓ Successfully processed {len(processed_7T)} 7T scans")

## Step 5.4: N4 Bias Field Correction

Apply N4ITK bias field correction to remove intensity inhomogeneities.

In [ ]:
def n4_bias_correction_sitk(input_path: Path, output_path: Path, 
                            mask_path: Path = None,
                            num_iterations: int = 50,
                            convergence_threshold: float = 0.001) -> bool:
    """
    Apply N4 bias field correction using SimpleITK.
    More robust than command-line tools for automated processing.
    """
    try:
        # Read image
        input_image = sitk.ReadImage(str(input_path), sitk.sitkFloat32)
        
        # Read mask if provided
        if mask_path and mask_path.exists():
            mask_image = sitk.ReadImage(str(mask_path), sitk.sitkUInt8)
        else:
            # Create a simple mask (everything non-zero)
            mask_image = sitk.BinaryThreshold(input_image, 0, 0, 0, 1)
        
        # Set up N4 bias field correction
        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        corrector.SetMaximumNumberOfIterations([num_iterations] * 4)
        corrector.SetConvergenceThreshold(convergence_threshold)
        
        # Execute correction
        output_image = corrector.Execute(input_image, mask_image)
        
        # Write output
        sitk.WriteImage(output_image, str(output_path))
        
        return True
    except Exception as e:
        print(f"N4 correction failed for {input_path}: {e}")
        return False

def process_n4_correction(processed_items: List[Dict], category: str) -> List[Dict]:
    """
    Apply N4 bias correction to all processed scans.
    """
    corrected_items = []
    
    output_dir = PROCESSED_DIR / category / "n4_corrected"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\nProcessing {len(processed_items)} {category} scans...")
    
    for item in tqdm(processed_items, desc=f"N4 correction {category}"):
        subject = item["subject"]
        brain_file = item["brain_file"]
        mask_file = item.get("mask_file")
        
        # Create output paths
        subj_dir = output_dir / subject
        subj_dir.mkdir(parents=True, exist_ok=True)
        
        output_file = subj_dir / f"{brain_file.stem}_n4.nii.gz"
        
        # Skip if already processed
        if output_file.exists():
            corrected_items.append({
                **item,
                "n4_file": output_file
            })
            continue
        
        # Apply N4 correction
        success = n4_bias_correction_sitk(brain_file, output_file, mask_file)
        
        if success:
            corrected_items.append({
                **item,
                "n4_file": output_file
            })
        else:
            print(f"Failed to process: {brain_file}")
    
    return corrected_items

# Process both 3T and 7T data
print("="*60)
print("N4 BIAS FIELD CORRECTION")
print("="*60)

corrected_3T = process_n4_correction(processed_3T, "3T")
corrected_7T = process_n4_correction(processed_7T, "7T")

print(f"\n✓ Successfully processed {len(corrected_3T)} 3T scans")
print(f"✓ Successfully processed {len(corrected_7T)} 7T scans")

## Step 5.5: Registration - Align 3T to 7T Space

Use ANTs-based registration (rigid + affine + SyN deformable) for voxel-wise alignment.

**Note:** This implementation uses SimpleITK for registration. For best results, use ANTs if available.

In [ ]:
def register_3T_to_7T_sitk(moving_path: Path, fixed_path: Path, 
                           output_path: Path,
                           transform_path: Path = None) -> bool:
    """
    Register 3T image to 7T space using SimpleITK multi-stage registration.
    Performs: Rigid -> Affine -> Deformable (B-spline)
    """
    try:
        # Read images
        fixed_image = sitk.ReadImage(str(fixed_path), sitk.sitkFloat32)
        moving_image = sitk.ReadImage(str(moving_path), sitk.sitkFloat32)
        
        # Initialize registration method
        registration = sitk.ImageRegistrationMethod()
        
        # Similarity metric
        registration.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
        registration.SetMetricSamplingStrategy(registration.RANDOM)
        registration.SetMetricSamplingPercentage(0.01)
        
        # Interpolator
        registration.SetInterpolator(sitk.sitkLinear)
        
        # Optimizer settings
        registration.SetOptimizerAsGradientDescent(
            learningRate=1.0,
            numberOfIterations=100,
            convergenceMinimumValue=1e-6,
            convergenceWindowSize=10
        )
        registration.SetOptimizerScalesFromPhysicalShift()
        
        # Multi-resolution framework
        registration.SetShrinkFactorsPerLevel(shrinkFactors=[4, 2, 1])
        registration.SetSmoothingSigmasPerLevel(smoothingSigmas=[2, 1, 0])
        registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
        
        # Stage 1: Rigid registration
        print("  Stage 1: Rigid registration...")
        initial_transform = sitk.CenteredTransformInitializer(
            fixed_image, moving_image,
            sitk.Euler3DTransform(),
            sitk.CenteredTransformInitializerFilter.GEOMETRY
        )
        registration.SetInitialTransform(initial_transform, inPlace=False)
        rigid_transform = registration.Execute(fixed_image, moving_image)
        
        # Stage 2: Affine registration
        print("  Stage 2: Affine registration...")
        affine_transform = sitk.AffineTransform(3)
        affine_transform.SetMatrix(rigid_transform.GetMatrix())
        affine_transform.SetTranslation(rigid_transform.GetTranslation())
        affine_transform.SetCenter(rigid_transform.GetCenter())
        
        registration.SetInitialTransform(affine_transform, inPlace=False)
        registration.SetOptimizerAsGradientDescent(
            learningRate=1.0,
            numberOfIterations=100,
            convergenceMinimumValue=1e-6,
            convergenceWindowSize=10
        )
        final_transform = registration.Execute(fixed_image, moving_image)
        
        # Stage 3: Deformable registration (B-spline)
        print("  Stage 3: Deformable (B-spline) registration...")
        
        # Create B-spline transform
        transform_domain_mesh_size = [8] * moving_image.GetDimension()
        bspline_transform = sitk.BSplineTransformInitializer(
            fixed_image, transform_domain_mesh_size
        )
        
        # Composite transform: affine + b-spline
        composite_transform = sitk.CompositeTransform(3)
        composite_transform.AddTransform(final_transform)
        composite_transform.AddTransform(bspline_transform)
        
        registration.SetInitialTransform(composite_transform, inPlace=True)
        registration.SetOptimizerAsLBFGSB(
            gradientConvergenceTolerance=1e-5,
            numberOfIterations=50
        )
        
        # Execute final registration
        final_composite = registration.Execute(fixed_image, moving_image)
        
        # Apply transformation
        print("  Applying transformation...")
        resampler = sitk.ResampleImageFilter()
        resampler.SetReferenceImage(fixed_image)
        resampler.SetInterpolator(sitk.sitkLinear)
        resampler.SetDefaultPixelValue(0)
        resampler.SetTransform(final_composite)
        
        output_image = resampler.Execute(moving_image)
        
        # Save registered image
        sitk.WriteImage(output_image, str(output_path))
        
        # Save transform if requested
        if transform_path:
            sitk.WriteTransform(final_composite, str(transform_path))
        
        print(f"  ✓ Registration complete")
        return True
        
    except Exception as e:
        print(f"Registration failed: {e}")
        return False

def register_ants(moving_path: Path, fixed_path: Path, output_path: Path,
                  transform_prefix: Path = None) -> bool:
    """
    Register using ANTs (if available) - more robust than SimpleITK.
    """
    try:
        # Check if ANTs is available
        result = subprocess.run(["antsRegistrationSyN.sh"], 
                              capture_output=True, timeout=5)
        ants_available = "Usage" in result.stderr.decode() or "Usage" in result.stdout.decode()
    except:
        ants_available = False
    
    if not ants_available:
        print("ANTs not available, using SimpleITK...")
        return register_3T_to_7T_sitk(moving_path, fixed_path, output_path)
    
    # Use ANTs
    prefix = transform_prefix or output_path.parent / f"{output_path.stem}_"
    
    cmd = [
        "antsRegistrationSyN.sh",
        "-d", "3",
        "-f", str(fixed_path),
        "-m", str(moving_path),
        "-o", str(prefix),
        "-t", "s",  # SyN (deformable)
        "-n", "4"   # 4 threads
    ]
    
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        
        # Rename output to match expected name
        warped_file = Path(str(prefix) + "Warped.nii.gz")
        if warped_file.exists():
            shutil.move(warped_file, output_path)
        
        return True
    except Exception as e:
        print(f"ANTs registration failed: {e}")
        return False

def match_and_register_pairs(items_3T: List[Dict], items_7T: List[Dict]) -> List[Dict]:
    """
    Match 3T and 7T scans by subject and register them.
    """
    # Group by subject
    subjects_3T = {}
    for item in items_3T:
        subj = item["subject"]
        if subj not in subjects_3T:
            subjects_3T[subj] = []
        subjects_3T[subj].append(item)
    
    subjects_7T = {}
    for item in items_7T:
        subj = item["subject"]
        if subj not in subjects_7T:
            subjects_7T[subj] = []
        subjects_7T[subj].append(item)
    
    # Find matching pairs
    matched_pairs = []
    for subj in subjects_3T:
        if subj in subjects_7T:
            # For simplicity, take first scan from each
            for item_3T in subjects_3T[subj]:
                for item_7T in subjects_7T[subj]:
                    matched_pairs.append({
                        "subject": subj,
                        "3T": item_3T,
                        "7T": item_7T
                    })
    
    print(f"Found {len(matched_pairs)} matched pairs")
    
    # Register each pair
    registered_pairs = []
    output_dir = PROCESSED_DIR / "registered"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    for pair in tqdm(matched_pairs, desc="Registering 3T to 7T"):
        subject = pair["subject"]
        moving_file = pair["3T"]["n4_file"]
        fixed_file = pair["7T"]["n4_file"]
        
        # Create output paths
        subj_dir = output_dir / subject
        subj_dir.mkdir(parents=True, exist_ok=True)
        
        output_file = subj_dir / f"{moving_file.stem}_registered_to_7T.nii.gz"
        transform_file = subj_dir / f"{moving_file.stem}_transform.txt"
        
        # Skip if already processed
        if output_file.exists():
            registered_pairs.append({
                **pair,
                "registered_3T": output_file,
                "transform": transform_file if transform_file.exists() else None
            })
            continue
        
        # Perform registration
        print(f"\nRegistering {subject}...")
        success = register_ants(moving_file, fixed_file, output_file, transform_file)
        
        if success:
            registered_pairs.append({
                **pair,
                "registered_3T": output_file,
                "transform": transform_file if transform_file.exists() else None
            })
        else:
            print(f"Failed to register: {subject}")
    
    return registered_pairs

# Match and register pairs
print("="*60)
print("REGISTRATION: 3T → 7T")
print("="*60)

registered_pairs = match_and_register_pairs(corrected_3T, corrected_7T)

print(f"\n✓ Successfully registered {len(registered_pairs)} pairs")

## Step 5.6: Resolution Harmonization

Resample 3T images to match 7T resolution (typically upsampling from ~1.0mm to ~0.7mm).

In [ ]:
def get_voxel_spacing(nifti_path: Path) -> Tuple[float, float, float]:
    """
    Get voxel spacing from NIfTI file.
    """
    img = nib.load(nifti_path)
    return tuple(img.header.get_zooms()[:3])

def resample_to_target_spacing(input_path: Path, output_path: Path,
                                target_spacing: Tuple[float, float, float],
                                interpolation: str = "linear") -> bool:
    """
    Resample image to target spacing using SimpleITK.
    """
    try:
        # Read image
        image = sitk.ReadImage(str(input_path))
        
        # Get original spacing
        original_spacing = image.GetSpacing()
        original_size = image.GetSize()
        
        # Calculate new size
        new_size = [
            int(round(original_size[i] * (original_spacing[i] / target_spacing[i])))
            for i in range(3)
        ]
        
        # Set up resampler
        resampler = sitk.ResampleImageFilter()
        resampler.SetOutputSpacing(target_spacing)
        resampler.SetSize(new_size)
        resampler.SetOutputDirection(image.GetDirection())
        resampler.SetOutputOrigin(image.GetOrigin())
        resampler.SetTransform(sitk.Transform())
        resampler.SetDefaultPixelValue(0)
        
        # Set interpolator
        if interpolation == "linear":
            resampler.SetInterpolator(sitk.sitkLinear)
        elif interpolation == "bspline":
            resampler.SetInterpolator(sitk.sitkBSpline)
        elif interpolation == "nearest":
            resampler.SetInterpolator(sitk.sitkNearestNeighbor)
        else:
            resampler.SetInterpolator(sitk.sitkLinear)
        
        # Resample
        resampled = resampler.Execute(image)
        
        # Save
        sitk.WriteImage(resampled, str(output_path))
        
        return True
        
    except Exception as e:
        print(f"Resampling failed: {e}")
        return False

def harmonize_resolution(registered_pairs: List[Dict]) -> List[Dict]:
    """
    Harmonize resolution of all registered pairs to match 7T spacing.
    """
    harmonized_pairs = []
    output_dir = PROCESSED_DIR / "harmonized"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("\nHarmonizing resolutions...")
    
    for pair in tqdm(registered_pairs, desc="Resampling to common grid"):
        subject = pair["subject"]
        
        # Get 7T spacing (target)
        target_7T_file = pair["7T"]["n4_file"]
        target_spacing = get_voxel_spacing(target_7T_file)
        
        # Get registered 3T file
        registered_3T_file = pair["registered_3T"]
        
        # Create output paths
        subj_dir = output_dir / subject
        subj_dir.mkdir(parents=True, exist_ok=True)
        
        output_3T = subj_dir / f"{registered_3T_file.stem}_resampled.nii.gz"
        output_7T = subj_dir / f"{target_7T_file.stem}_resampled.nii.gz"
        
        # Skip if already processed
        if output_3T.exists() and output_7T.exists():
            harmonized_pairs.append({
                **pair,
                "harmonized_3T": output_3T,
                "harmonized_7T": output_7T,
                "spacing": target_spacing
            })
            continue
        
        # Resample 3T to target spacing
        print(f"\n  {subject}: Resampling 3T to {target_spacing}")
        success_3T = resample_to_target_spacing(
            registered_3T_file, output_3T, target_spacing, "linear"
        )
        
        # Also ensure 7T is at exact target spacing (in case of minor variations)
        success_7T = resample_to_target_spacing(
            target_7T_file, output_7T, target_spacing, "linear"
        )
        
        if success_3T and success_7T:
            harmonized_pairs.append({
                **pair,
                "harmonized_3T": output_3T,
                "harmonized_7T": output_7T,
                "spacing": target_spacing
            })
        else:
            print(f"Failed to harmonize: {subject}")
    
    return harmonized_pairs

# Harmonize resolutions
print("="*60)
print("RESOLUTION HARMONIZATION")
print("="*60)

harmonized_pairs = harmonize_resolution(registered_pairs)

print(f"\n✓ Successfully harmonized {len(harmonized_pairs)} pairs")

# Print spacing information
if harmonized_pairs:
    example_pair = harmonized_pairs[0]
    print(f"\nTarget spacing: {example_pair['spacing']}")
    print(f"  (typically 7T: ~0.7mm³, 3T upsampled to match)")

## Step 5.7: Intensity Normalization

Apply z-score normalization within brain mask. Critical for GAN training consistency.

In [ ]:
def zscore_normalize(input_path: Path, output_path: Path, 
                     mask_path: Path = None) -> bool:
    """
    Apply z-score normalization within brain mask.
    Normalizes to mean=0, std=1.
    """
    try:
        # Load image
        img = nib.load(input_path)
        data = img.get_fdata()
        
        # Load or create mask
        if mask_path and mask_path.exists():
            mask_img = nib.load(mask_path)
            mask = mask_img.get_fdata() > 0
        else:
            # Create mask from non-zero voxels
            mask = data > 0
        
        # Get brain voxels
        brain_voxels = data[mask]
        
        if len(brain_voxels) == 0:
            print(f"Warning: No brain voxels found in {input_path}")
            return False
        
        # Calculate statistics
        mean_val = np.mean(brain_voxels)
        std_val = np.std(brain_voxels)
        
        # Normalize
        if std_val > 0:
            data_norm = (data - mean_val) / std_val
            # Keep only brain region
            data_norm = data_norm * mask
        else:
            print(f"Warning: std is 0 for {input_path}")
            data_norm = data * mask
        
        # Save normalized image
        norm_img = nib.Nifti1Image(data_norm, img.affine, img.header)
        nib.save(norm_img, output_path)
        
        return True
        
    except Exception as e:
        print(f"Normalization failed: {e}")
        return False

def percentile_normalize(input_path: Path, output_path: Path,
                         lower_percentile: float = 1.0,
                         upper_percentile: float = 99.0) -> bool:
    """
    Normalize intensities using percentile clipping and scaling to [0, 1].
    Alternative to z-score, useful for visualization.
    """
    try:
        img = nib.load(input_path)
        data = img.get_fdata()
        
        # Get non-zero values
        nonzero_data = data[data > 0]
        
        if len(nonzero_data) == 0:
            return False
        
        # Calculate percentiles
        lower_val = np.percentile(nonzero_data, lower_percentile)
        upper_val = np.percentile(nonzero_data, upper_percentile)
        
        # Clip and normalize
        data_norm = np.clip(data, lower_val, upper_val)
        data_norm = (data_norm - lower_val) / (upper_val - lower_val + 1e-8)
        data_norm = data_norm * (data > 0)  # Preserve background
        
        # Save
        norm_img = nib.Nifti1Image(data_norm, img.affine, img.header)
        nib.save(norm_img, output_path)
        
        return True
        
    except Exception as e:
        print(f"Percentile normalization failed: {e}")
        return False

def normalize_intensities(harmonized_pairs: List[Dict]) -> List[Dict]:
    """
    Apply intensity normalization to all harmonized pairs.
    """
    normalized_pairs = []
    output_dir = PROCESSED_DIR / "normalized"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("\nNormalizing intensities...")
    
    for pair in tqdm(harmonized_pairs, desc="Z-score normalization"):
        subject = pair["subject"]
        
        # Get mask if available
        mask_3T = pair["3T"].get("mask_file")
        mask_7T = pair["7T"].get("mask_file")
        
        # Get harmonized files
        harmonized_3T = pair["harmonized_3T"]
        harmonized_7T = pair["harmonized_7T"]
        
        # Create output paths
        subj_dir = output_dir / subject
        subj_dir.mkdir(parents=True, exist_ok=True)
        
        output_3T = subj_dir / f"{harmonized_3T.stem}_norm.nii.gz"
        output_7T = subj_dir / f"{harmonized_7T.stem}_norm.nii.gz"
        
        # Skip if already processed
        if output_3T.exists() and output_7T.exists():
            normalized_pairs.append({
                **pair,
                "normalized_3T": output_3T,
                "normalized_7T": output_7T
            })
            continue
        
        # Normalize 3T
        success_3T = zscore_normalize(harmonized_3T, output_3T, mask_3T)
        
        # Normalize 7T
        success_7T = zscore_normalize(harmonized_7T, output_7T, mask_7T)
        
        if success_3T and success_7T:
            normalized_pairs.append({
                **pair,
                "normalized_3T": output_3T,
                "normalized_7T": output_7T
            })
        else:
            print(f"Failed to normalize: {subject}")
    
    return normalized_pairs

# Normalize intensities
print("="*60)
print("INTENSITY NORMALIZATION")
print("="*60)

normalized_pairs = normalize_intensities(harmonized_pairs)

print(f"\n✓ Successfully normalized {len(normalized_pairs)} pairs")
print("\nNormalization: Z-score (mean=0, std=1) within brain mask")

## Step 5.8: Tissue Segmentation (Optional but Recommended)

Generate GM/WM/CSF segmentation masks using FastSurfer or SynthSeg.

This enables:
- Topology-aware loss functions
- Dice metrics for clinical validation
- Region-specific analysis

In [ ]:
def segment_tissues_synthseg(input_path: Path, output_seg_path: Path,
                             output_vol_path: Path = None) -> bool:
    """
    Perform tissue segmentation using SynthSeg (if available).
    Falls back to simple thresholding if not available.
    """
    # Check if SynthSeg/mri_synthseg is available
    try:
        result = subprocess.run(["mri_synthseg", "--help"], 
                              capture_output=True, timeout=5)
        synthseg_available = result.returncode == 0
    except:
        synthseg_available = False
    
    if synthseg_available:
        # Use SynthSeg
        cmd = ["mri_synthseg", "--i", str(input_path), "--o", str(output_seg_path)]
        if output_vol_path:
            cmd.extend(["--vol", str(output_vol_path)])
        
        try:
            subprocess.run(cmd, check=True, capture_output=True)
            return True
        except subprocess.CalledProcessError as e:
            print(f"SynthSeg failed: {e}")
            return False
    else:
        # Fallback: Simple intensity-based segmentation
        print("SynthSeg not available, using intensity-based segmentation...")
        return segment_tissues_fallback(input_path, output_seg_path)

def segment_tissues_fallback(input_path: Path, output_seg_path: Path) -> bool:
    """
    Fallback tissue segmentation using intensity thresholding.
    Labels: 0=Background, 1=CSF, 2=GM, 3=WM
    """
    try:
        # Load image
        img = nib.load(input_path)
        data = img.get_fdata()
        
        # Normalize for thresholding
        brain_mask = data > 0
        if np.sum(brain_mask) == 0:
            return False
        
        brain_data = data[brain_mask]
        mean_val = np.mean(brain_data)
        std_val = np.std(brain_data)
        
        # Initialize segmentation
        seg = np.zeros_like(data, dtype=np.uint8)
        
        # Simple intensity-based classification
        # CSF: low intensity
        csf_mask = (data > 0) & (data < mean_val - 0.5 * std_val)
        seg[csf_mask] = 1
        
        # WM: high intensity
        wm_mask = data > mean_val + 0.5 * std_val
        seg[wm_mask] = 3
        
        # GM: medium intensity
        gm_mask = (data >= mean_val - 0.5 * std_val) & (data <= mean_val + 0.5 * std_val) & (data > 0)
        seg[gm_mask] = 2
        
        # Save segmentation
        seg_img = nib.Nifti1Image(seg, img.affine, img.header)
        nib.save(seg_img, output_seg_path)
        
        return True
        
    except Exception as e:
        print(f"Fallback segmentation failed: {e}")
        return False

def create_tissue_masks(seg_path: Path, output_dir: Path) -> Dict[str, Path]:
    """
    Create separate binary masks for each tissue type.
    """
    try:
        seg_img = nib.load(seg_path)
        seg_data = seg_img.get_fdata()
        
        masks = {}
        tissue_types = {
            "csf": 1,
            "gm": 2,
            "wm": 3
        }
        
        for tissue_name, label in tissue_types.items():
            mask = (seg_data == label).astype(np.uint8)
            mask_path = output_dir / f"mask_{tissue_name}.nii.gz"
            mask_img = nib.Nifti1Image(mask, seg_img.affine, seg_img.header)
            nib.save(mask_img, mask_path)
            masks[tissue_name] = mask_path
        
        return masks
        
    except Exception as e:
        print(f"Failed to create tissue masks: {e}")
        return {}

def segment_all_scans(normalized_pairs: List[Dict]) -> List[Dict]:
    """
    Perform tissue segmentation on all normalized scans.
    """
    segmented_pairs = []
    output_dir = PROCESSED_DIR / "segmented"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("\nPerforming tissue segmentation...")
    
    for pair in tqdm(normalized_pairs, desc="Tissue segmentation"):
        subject = pair["subject"]
        
        # Get normalized files
        norm_3T = pair["normalized_3T"]
        norm_7T = pair["normalized_7T"]
        
        # Create output paths
        subj_dir = output_dir / subject
        subj_dir.mkdir(parents=True, exist_ok=True)
        
        seg_3T = subj_dir / f"{norm_3T.stem}_seg.nii.gz"
        seg_7T = subj_dir / f"{norm_7T.stem}_seg.nii.gz"
        
        # Skip if already processed
        if seg_3T.exists() and seg_7T.exists():
            # Create tissue masks
            masks_3T = create_tissue_masks(seg_3T, subj_dir / "3T_masks")
            masks_7T = create_tissue_masks(seg_7T, subj_dir / "7T_masks")
            
            segmented_pairs.append({
                **pair,
                "seg_3T": seg_3T,
                "seg_7T": seg_7T,
                "masks_3T": masks_3T,
                "masks_7T": masks_7T
            })
            continue
        
        # Segment 3T
        success_3T = segment_tissues_synthseg(norm_3T, seg_3T)
        
        # Segment 7T
        success_7T = segment_tissues_synthseg(norm_7T, seg_7T)
        
        if success_3T and success_7T:
            # Create tissue masks
            (subj_dir / "3T_masks").mkdir(exist_ok=True)
            (subj_dir / "7T_masks").mkdir(exist_ok=True)
            
            masks_3T = create_tissue_masks(seg_3T, subj_dir / "3T_masks")
            masks_7T = create_tissue_masks(seg_7T, subj_dir / "7T_masks")
            
            segmented_pairs.append({
                **pair,
                "seg_3T": seg_3T,
                "seg_7T": seg_7T,
                "masks_3T": masks_3T,
                "masks_7T": masks_7T
            })
        else:
            print(f"Failed to segment: {subject}")
    
    return segmented_pairs

# Perform segmentation
print("="*60)
print("TISSUE SEGMENTATION (OPTIONAL)")
print("="*60)

segmented_pairs = segment_all_scans(normalized_pairs)

print(f"\n✓ Successfully segmented {len(segmented_pairs)} pairs")
print("\nTissue labels: 1=CSF, 2=GM, 3=WM")

## Pipeline Summary and Final Dataset Export

Generate summary statistics and export the final preprocessed dataset.

In [ ]:
def export_final_dataset(pairs: List[Dict], output_dir: Path) -> None:
    """
    Export final preprocessed dataset in organized structure for GAN training.
    """
    # Create final directory structure
    final_3T_dir = output_dir / "final" / "3T"
    final_7T_dir = output_dir / "final" / "7T"
    final_3T_dir.mkdir(parents=True, exist_ok=True)
    final_7T_dir.mkdir(parents=True, exist_ok=True)
    
    manifest = []
    
    print("\nExporting final dataset...")
    
    for pair in tqdm(pairs, desc="Copying final files"):
        subject = pair["subject"]
        
        # Final normalized files (these are the ones to use for training)
        src_3T = pair["normalized_3T"]
        src_7T = pair["normalized_7T"]
        
        # Destination paths with clean naming
        dst_3T = final_3T_dir / f"{subject}_3T.nii.gz"
        dst_7T = final_7T_dir / f"{subject}_7T.nii.gz"
        
        # Copy files
        if not dst_3T.exists():
            shutil.copy2(src_3T, dst_3T)
        if not dst_7T.exists():
            shutil.copy2(src_7T, dst_7T)
        
        # Add to manifest
        manifest.append({
            "subject": subject,
            "3T_file": str(dst_3T.relative_to(output_dir)),
            "7T_file": str(dst_7T.relative_to(output_dir)),
            "spacing": pair.get("spacing", "unknown"),
            "has_segmentation": "seg_3T" in pair
        })
    
    # Save manifest as JSON
    manifest_path = output_dir / "final" / "dataset_manifest.json"
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    
    # Save manifest as CSV for easy viewing
    manifest_df = pd.DataFrame(manifest)
    manifest_csv = output_dir / "final" / "dataset_manifest.csv"
    manifest_df.to_csv(manifest_csv, index=False)
    
    print(f"\n✓ Dataset exported to: {output_dir / 'final'}")
    print(f"✓ Manifest saved: {manifest_path}")
    print(f"✓ Total pairs: {len(manifest)}")
    
    return manifest

# Export final dataset
print("="*60)
print("EXPORTING FINAL DATASET")
print("="*60)

# Use segmented pairs if available, otherwise normalized pairs
final_pairs = segmented_pairs if segmented_pairs else normalized_pairs
manifest = export_final_dataset(final_pairs, PROCESSED_DIR)

# Display summary
print(f"\n{'='*60}")
print(f"PREPROCESSING PIPELINE COMPLETE")
print(f"{'='*60}")
print(f"✓ Processed subjects: {len(manifest)}")
print(f"✓ Output directory: {PROCESSED_DIR / 'final'}")
print(f"\nDataset ready for GAN training!")
print(f"\nUse the files in:")
print(f"  - {PROCESSED_DIR / 'final' / '3T'}  (input)")
print(f"  - {PROCESSED_DIR / 'final' / '7T'}  (target)")

## Visualization: Quality Control

Visualize sample slices from preprocessed data to verify quality.

In [ ]:
def visualize_preprocessing_results(pair: Dict, slice_idx: int = None):
    """
    Visualize preprocessing results for a single subject.
    Shows: Original, Skull-stripped, N4-corrected, Registered, Normalized
    """
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    subject = pair["subject"]
    fig.suptitle(f"Preprocessing Pipeline: {subject}", fontsize=16, fontweight='bold')
    
    # Load images
    images_3T = {
        "Original": pair["3T"]["nii_file"],
        "Skull-stripped": pair["3T"]["brain_file"],
        "N4 Corrected": pair["3T"]["n4_file"],
        "Normalized": pair["normalized_3T"]
    }
    
    images_7T = {
        "Original": pair["7T"]["nii_file"],
        "Skull-stripped": pair["7T"]["brain_file"],
        "N4 Corrected": pair["7T"]["n4_file"],
        "Normalized": pair["normalized_7T"]
    }
    
    # Get middle slice if not specified
    if slice_idx is None:
        img = nib.load(pair["normalized_3T"])
        slice_idx = img.shape[2] // 2
    
    # Plot 3T images
    for idx, (title, img_path) in enumerate(images_3T.items()):
        img = nib.load(img_path)
        data = img.get_fdata()
        
        axes[0, idx].imshow(data[:, :, slice_idx].T, cmap='gray', origin='lower')
        axes[0, idx].set_title(f"3T: {title}", fontsize=12, fontweight='bold')
        axes[0, idx].axis('off')
    
    # Plot 7T images
    for idx, (title, img_path) in enumerate(images_7T.items()):
        img = nib.load(img_path)
        data = img.get_fdata()
        
        axes[1, idx].imshow(data[:, :, slice_idx].T, cmap='gray', origin='lower')
        axes[1, idx].set_title(f"7T: {title}", fontsize=12, fontweight='bold')
        axes[1, idx].axis('off')
    
    plt.tight_layout()
    plt.show()

def visualize_paired_comparison(pair: Dict, slice_idx: int = None):
    """
    Compare final 3T and 7T images side-by-side.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    subject = pair["subject"]
    fig.suptitle(f"Final Paired Images: {subject}", fontsize=16, fontweight='bold')
    
    # Load final images
    img_3T = nib.load(pair["normalized_3T"])
    img_7T = nib.load(pair["normalized_7T"])
    
    data_3T = img_3T.get_fdata()
    data_7T = img_7T.get_fdata()
    
    # Get middle slice if not specified
    if slice_idx is None:
        slice_idx = data_3T.shape[2] // 2
    
    slice_3T = data_3T[:, :, slice_idx].T
    slice_7T = data_7T[:, :, slice_idx].T
    
    # 3T image
    axes[0].imshow(slice_3T, cmap='gray', origin='lower')
    axes[0].set_title('3T (Input)', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # 7T image
    axes[1].imshow(slice_7T, cmap='gray', origin='lower')
    axes[1].set_title('7T (Target)', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    # Difference map
    diff = np.abs(slice_7T - slice_3T)
    im = axes[2].imshow(diff, cmap='hot', origin='lower')
    axes[2].set_title('Absolute Difference', fontsize=14, fontweight='bold')
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046)
    
    plt.tight_layout()
    plt.show()

# Visualize first subject
if final_pairs:
    print("Visualizing preprocessing results for first subject...")
    visualize_preprocessing_results(final_pairs[0])
    
    print("\nVisualizing paired comparison...")
    visualize_paired_comparison(final_pairs[0])
else:
    print("No processed pairs available for visualization")

## Additional Utilities and Statistics

In [ ]:
def compute_dataset_statistics(pairs: List[Dict]) -> pd.DataFrame:
    """
    Compute comprehensive statistics for the preprocessed dataset.
    """
    stats_list = []
    
    for pair in tqdm(pairs, desc="Computing statistics"):
        subject = pair["subject"]
        
        # Load normalized images
        img_3T = nib.load(pair["normalized_3T"])
        img_7T = nib.load(pair["normalized_7T"])
        
        data_3T = img_3T.get_fdata()
        data_7T = img_7T.get_fdata()
        
        # Get brain mask (non-zero voxels)
        mask_3T = data_3T > 0
        mask_7T = data_7T > 0
        
        brain_3T = data_3T[mask_3T]
        brain_7T = data_7T[mask_7T]
        
        stats = {
            "Subject": subject,
            "3T_Shape": str(data_3T.shape),
            "7T_Shape": str(data_7T.shape),
            "3T_Spacing": str(img_3T.header.get_zooms()[:3]),
            "7T_Spacing": str(img_7T.header.get_zooms()[:3]),
            "3T_Brain_Voxels": len(brain_3T),
            "7T_Brain_Voxels": len(brain_7T),
            "3T_Mean": np.mean(brain_3T),
            "3T_Std": np.std(brain_3T),
            "3T_Min": np.min(brain_3T),
            "3T_Max": np.max(brain_3T),
            "7T_Mean": np.mean(brain_7T),
            "7T_Std": np.std(brain_7T),
            "7T_Min": np.min(brain_7T),
            "7T_Max": np.max(brain_7T)
        }
        
        stats_list.append(stats)
    
    return pd.DataFrame(stats_list)

def save_preprocessing_summary(pairs: List[Dict], output_path: Path):
    """
    Save a comprehensive preprocessing summary report.
    """
    summary = {
        "pipeline_version": "1.0",
        "processing_date": pd.Timestamp.now().isoformat(),
        "total_subjects": len(pairs),
        "preprocessing_steps": [
            "BIDS validation",
            "Field strength separation (3T/7T)",
            "Skull stripping (SynthStrip/fallback)",
            "N4 bias field correction",
            "Registration (3T → 7T space)",
            "Resolution harmonization",
            "Z-score intensity normalization",
            "Tissue segmentation (optional)"
        ],
        "output_structure": {
            "3T": "Input images for GAN",
            "7T": "Target images for GAN",
            "normalized": "Intermediate normalized images",
            "registered": "Intermediate registered images",
            "segmented": "Tissue segmentation masks (if generated)"
        }
    }
    
    with open(output_path, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"✓ Summary saved to: {output_path}")

# Compute and display statistics
if final_pairs:
    print("\n" + "="*60)
    print("DATASET STATISTICS")
    print("="*60)
    
    stats_df = compute_dataset_statistics(final_pairs)
    
    # Display summary statistics
    print(f"\nDataset Overview:")
    print(f"  Total subjects: {len(stats_df)}")
    print(f"\nIntensity Statistics (Z-score normalized):")
    print(f"  3T Mean: {stats_df['3T_Mean'].mean():.4f} ± {stats_df['3T_Mean'].std():.4f}")
    print(f"  3T Std:  {stats_df['3T_Std'].mean():.4f} ± {stats_df['3T_Std'].std():.4f}")
    print(f"  7T Mean: {stats_df['7T_Mean'].mean():.4f} ± {stats_df['7T_Mean'].std():.4f}")
    print(f"  7T Std:  {stats_df['7T_Std'].mean():.4f} ± {stats_df['7T_Std'].std():.4f}")
    
    # Save statistics
    stats_csv = PROCESSED_DIR / "final" / "dataset_statistics.csv"
    stats_df.to_csv(stats_csv, index=False)
    print(f"\n✓ Full statistics saved to: {stats_csv}")
    
    # Save preprocessing summary
    summary_path = PROCESSED_DIR / "final" / "preprocessing_summary.json"
    save_preprocessing_summary(final_pairs, summary_path)
    
    # Display first few entries
    print(f"\nFirst 3 subjects:")
    print(stats_df.head(3).to_string(index=False))
else:
    print("No data available for statistics computation")

## Next Steps

The preprocessing pipeline is complete! Your data is now ready for GAN training.

### What We've Accomplished:
✅ **BIDS Validation** - Verified dataset structure  
✅ **Field Strength Separation** - Organized 3T and 7T scans  
✅ **Skull Stripping** - Removed non-brain tissue  
✅ **Bias Field Correction** - Applied N4 correction  
✅ **Registration** - Aligned 3T to 7T space with voxel-wise correspondence  
✅ **Resolution Harmonization** - Resampled to common grid  
✅ **Intensity Normalization** - Z-score normalized (mean=0, std=1)  
✅ **Tissue Segmentation** - Generated GM/WM/CSF masks (optional)  

### Ready for Training:
- **Input (3T)**: `data/processed/final/3T/`
- **Target (7T)**: `data/processed/final/7T/`
- **Manifest**: `data/processed/final/dataset_manifest.json`

### Next: Train Your GAN
```python
# Use the training script
python scripts/train_gan.py --config configs/gan_config.yaml

# Or use the training notebook
# notebooks/gan_training_notebook.ipynb
```

### Quality Assurance Checklist:
- [ ] Verify voxel-wise alignment between 3T and 7T
- [ ] Check intensity distributions are normalized
- [ ] Ensure no registration artifacts
- [ ] Validate brain masks cover full brain
- [ ] Confirm matching subject counts in 3T and 7T folders